# GRPO, visually — the whole pipeline, one cell per concept

Companion to `scripts/train/` (verifier → baseline_probe → grpo). Run top-to-bottom
with the **Python (grpo-trl)** kernel. Only the marked cells need the GPU / the real
model; everything else is CPU and instant.

**What each section maps to in the real pipeline:**

| Notebook section | Pipeline piece |
|---|---|
| 1. Data formatting | `build_coldstart.py` output → what `sft.py`/`grpo.py` actually feed the model |
| 2. The verifier | `verifier.py` = the reward function = the ONLY signal GRPO optimizes |
| 3. Group advantage math | the core of `GRPOTrainer` — nothing more than this formula |
| 4. Real rollouts | what `grpo.py --num-generations 8` does per prompt, visualized |
| 5. Live curves | `report_to="trackio"` in `grpo.py` — the real run's dashboard |

**External visuals worth your time** (open alongside this notebook):
- [Machine Learning Visualized](https://ml-visualized.com/) (Philip's fork: PhilipEriksson/machine-learning-visualized) — classical ML from scratch, training-loop convergence visualizations; the genre this notebook borrows. Notebooks live in the sibling gavinkhung repos.\n- [Transformer Explainer](https://poloclub.github.io/transformer-explainer/) — interactive, every layer animates; the best 10 minutes you'll spend\n- [The Illustrated Transformer](https://jalammar.github.io/illustrated-transformer/) — the classic diagrams
- [BertViz](https://bertviz.readthedocs.io/) — attention head visualization
- [TRL GRPOTrainer docs](https://huggingface.co/docs/trl/main/en/grpo_trainer) — every knob explained, with the logged-metrics table
- Karpathy's [Intro to LLMs / Let's build GPT](https://www.youtube.com/@AndrejKarpathy) — tokenization & attention from scratch

## 1. Data formatting — what the model actually sees

`train.jsonl` rows look like `{"prompt", "answer", "solution"}` — but a model never sees JSON.
The trainer renders each row through the **chat template**, tokenizes, and feeds token ids.
Look at exactly what text and which tokens come out — this is the layer where most
data bugs hide (template mismatches, missing generation prompt, stray tokens).

In [1]:
import json, pathlib

rows = [json.loads(l) for l in open(pathlib.Path('../datasets/processed/gsm8k-coldstart/train.jsonl').resolve())]
print(f"{len(rows)} rows | keys: {sorted(rows[0])}\n")
ex = rows[0]
print("RAW prompt field:\n" + "-"*70)
print(ex["prompt"][:400])
print("-"*70 + "\nRAW solution (CoT the SFT stage would teach):")
print(ex["solution"][:400] + f"\n\n<answer>{ex['answer']}</answer>")  # what sft.py to_chat builds

FileNotFoundError: [Errno 2] No such file or directory: '/home/p5090/Workspace/datasets/processed/gsm8k-coldstart/train.jsonl'

In [ ]:
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained("Qwen/Qwen3-0.6B")

SYSTEM = ("Solve the math problem step by step, then give the final numeric "
          "answer inside <answer></answer> tags.")

# Exactly what sft.py's to_chat() builds:
rendered = tok.apply_chat_template(
    [{"role": "system", "content": SYSTEM},
     {"role": "user", "content": ex["prompt"]}],
    tokenize=False, add_generation_prompt=True)
print(rendered)

In [ ]:
ids = tok.encode(rendered)
print(f"rendered prompt = {len(ids)} tokens\n")
# Token view — this is what crosses the GPU. Note how special tokens bracket everything.
for i in ids[:40]:
    print(f"{i:>7}  {tok.decode([i])!r}")
print("  ...")

## 2. The verifier — the ONLY teacher GRPO has

GRPO is RLVR: the reward is a *program*, not a model. Ours (`verifier.py`) is two
functions: `format_reward` (did it emit `<answer>…</answer>`?) and `accuracy_reward`
(does the extracted answer match gold, via `math_verify`).

Score some deliberately-written completions and watch the reward — if you can't
predict these scores, you don't yet understand what the run will optimize. This
function is also why reward hacking happens: the model will find ANY string that
passes it, not the "intended" behavior.

In [ ]:
import sys; sys.path.insert(0, '../scripts/train')
from verifier import ANSWER_RE, extract_answer, format_reward, accuracy_reward

completions = [
    "Let's think step by step. 120-75=45. <answer>45</answer>",   # correct + format
    "The answer is <answer>45</answer>",                            # format, no CoT
    "So the total is 45 minutes. <answer>44</answer>",              # wrong number
    "It takes 45 minutes in total.",                                # right-ish, NO tags
    "<answer>120 - 75 = 45 minutes</answer>",                       # format, not numeric
]
golds = ["45"] * len(completions)

f = format_reward(completions)
a = accuracy_reward(completions, answer=golds)
for c, fi, ai in zip(completions, f, a):
    print(f"format={fi:.1f}  accuracy={ai:.1f}   {c[:60]!r}")

## 3. The group advantage — GRPO's entire "algorithm" in one chart

There is no critic network in GRPO (that's PPO's expensive part). Instead: sample
**G completions per prompt**, score them, and grade each against its **group**:

**A_i = (r_i − mean(r)) / std(r)**

Below-average completions get pushed down, above-average pushed up — same prompt,
same policy, so the group IS the baseline. Run the cell: edit the fake rewards and
watch advantages re-center to zero mean, unit std. This is the whole trick.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def group_advantages(rewards):
    r = np.array(rewards, dtype=float)
    return (r - r.mean()) / (r.std() + 1e-8)   # same eps as TRL's default

rewards = [1.0, 0.0, 0.0, 1.0, 1.0, 0.0, 0.0, 0.0]  # 8 rollouts of ONE prompt
A = group_advantages(rewards)

fig, ax = plt.subplots(figsize=(8, 3))
colors = ['#2a9d8f' if a > 0 else '#e76f51' for a in A]
ax.bar(range(1, 9), A, color=colors)
ax.axhline(0, color='k', lw=1)
ax.set_xlabel("rollout #"); ax.set_ylabel("advantage A_i")
ax.set_title(f"rewards={rewards}\nmean={np.mean(rewards):.2f} -> advantages centered (green=reinforce, red=suppress)")
plt.show()

# Try: all-same rewards -> advantages all 0. THIS is why a 0%-pass-rate base model
# can't be GRPO'd (no signal) and why >50% wastes compute (advantages tiny).
print("all-correct group:", group_advantages([1.0]*8).round(3))

## 4. Real rollouts — the GRPO inner loop, live

*(GPU cell — :30000 must be stopped. ~2 min on first run.)*

This does by hand what `GRPOTrainer` does every step: sample G=8 completions for
one prompt at temperature 0.9, score with the real verifier, plot the group.
Every training step is just: this cell → advantage chart → weighted logprob update.

In [ ]:
from vllm import LLM, SamplingParams

PROMPT = rows[0]["prompt"]
GOLD = rows[0]["answer"]
messages = [{"role": "user",
             "content": f"{PROMPT}\n\nEnd your reply with the final numeric answer inside <answer></answer> tags."}]

llm = LLM(model="Qwen/Qwen3-0.6B", gpu_memory_utilization=0.6, max_model_len=2048)
sp = SamplingParams(temperature=0.9, max_tokens=512, n=8)
out = llm.generate([tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)], sp)[0]

texts = [o.text for o in out.outputs]
r_fmt = format_reward(texts)
r_acc = accuracy_reward(texts, answer=[GOLD])
r = np.array(r_fmt) + np.array(r_acc)   # grpo.py passes both funcs; TRL sums them

fig, ax = plt.subplots(figsize=(8, 3))
colors = ['#2a9d8f' if a > 0 else '#e76f51' for a in group_advantages(r)]
ax.bar(range(1, 9), group_advantages(r), color=colors)
ax.set_title(f"one real GRPO group | rewards={r.tolist()}"); ax.set_xlabel("rollout #")
plt.show()

print(f"gold answer: {GOLD}")
for t, rf, ra in zip(texts, r_fmt, r_acc):
    print(f"\n[fmt={rf:.0f} acc={ra:.0f}] {t[:150]!r}")

## 5. Watching the real run — trackio

`grpo.py` sets `report_to="trackio"`. When the real run starts, TRL serves a local
dashboard (link is printed in the first log lines) with, per step:

- `reward` / `rewards/format_reward` / `rewards/accuracy_reward` — **the curve that must go up**
- `completions/mean_length` — watch for degenerate long/short blowups
- `entropy` (if logged) — **collapse to ~0 = the model stopped exploring** (the classic GRPO failure)
- `clip_ratio`/`kl` — beta=0.0 means no KL here, so only clip matters

**Failure triage from the curves:**
| symptom | diagnosis |
|---|---|
| reward flat from step 0 | verifier never fires → re-run the probe; check format rate |
| reward jumps then flatlines at a hack | model found a verifier loophole → tighten verifier, log completions |
| entropy → 0 early | no KL (beta=0) + low temp → raise temperature / add small beta |
| loss NaN | LR too high for 0.6B or advantage std explosion |

Read TRL's logged-metrics table (link at top) for the full list.

## 6. Connect to the live run — read what the trainer actually wrote

The run writes checkpoints under `outputs/runs/`. Each `checkpoint-*/trainer_state.json`
contains `log_history`: the per-step metrics trackio charts, on disk where this notebook
can read them. This section is the bridge — run it *any time* (even mid-training; it
reads the latest checkpoint so far) to plot the run's real reward curve next to the
concepts above. No GPU needed, safe to run while GRPO trains.

In [ ]:
import json, pathlib, re

RUNS = pathlib.Path('../outputs/runs').resolve()

def latest_checkpoint(run_name):
    d = RUNS / run_name
    if not d.exists():
        return None
    ckpts = sorted(d.glob('checkpoint-*'), key=lambda p: int(p.name.split('-')[-1]))
    return ckpts[-1] if ckpts else None

for run in ['grpo-0.6b-grpo', 'grpo-0.6b-sft']:
    ck = latest_checkpoint(run)
    print(f"{run}: {ck if ck else 'no checkpoints yet'}")

In [ ]:
import matplotlib.pyplot as plt

ck = latest_checkpoint('grpo-0.6b-grpo')
if ck is None:
    print("No GRPO run yet. Start one (see section 5), then re-run this cell.")
else:
    hist = json.load(open(ck / 'trainer_state.json'))['log_history']
    steps = [h['step'] for h in hist if 'reward' in h]
    reward = [h['reward'] for h in hist if 'reward' in h]
    fig, axes = plt.subplots(1, 2, figsize=(11, 3))
    axes[0].plot(steps, reward, color='#2a9d8f')
    axes[0].set_title('reward (the curve that must go up)'); axes[0].set_xlabel('step')
    for key, color in [('rewards/format_reward/mean', '#e9c46a'), ('rewards/accuracy_reward/mean', '#e76f51')]:
        pts = [(h['step'], h[key]) for h in hist if key in h]
        if pts:
            axes[1].plot(*zip(*pts), color=color, label=key.split('/')[-1])
    axes[1].legend(); axes[1].set_title('reward components'); axes[1].set_xlabel('step')
    plt.tight_layout(); plt.show()
    print(f"checkpoint: {ck.name} | {len(steps)} logged steps")
    if steps and reward[-1] - reward[0] < 0.05 and len(steps) > 40:
        print("WARNING: reward flat after 40+ steps -> re-read section 4's dead-reward finding.")

**Second wire: `log_completions=True` in `grpo.py`** — TRL then logs 8 sample
completions per step into trackio, so you can *watch the model's text evolve* alongside
the curve (this is how you catch reward hacking early: the text starts gaming the
verifier before the numbers show it). One line, already added to the repo's grpo.py.
Trackio's local dashboard URL prints in the run's first log lines.

---
*Part of `ml-stack/scripts/train` — the writeup (README.md results + failures log) is the deliverable.*